# Processing the data (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

# Same as before
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# This is new
batch["labels"] = torch.tensor([1, 1])

optimizer = AdamW(model.parameters())
loss = model(**batch).loss
loss.backward()
optimizer.step()

In [ ]:
!pip install -U datasets huggingface_hub fsspec

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

In [ ]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

In [ ]:
raw_train_dataset.features

In [ ]:
#element 15 of training set
raw_datasets["train"][14]

In [ ]:
#element 87 of validation set
raw_datasets["validation"][86]

In [ ]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenized_sentences_1 = tokenizer(raw_datasets["train"]["sentence1"])
tokenized_sentences_2 = tokenizer(raw_datasets["train"]["sentence2"])

In [ ]:
inputs = tokenizer("This is the first sentence.", "This is the second one.")
inputs

In [ ]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"])

In [ ]:
#Take element 15 of the training set and tokenize the two sentences separately and as a pair.
tokenized_sentences_1 = tokenizer(raw_datasets["train"][14]["sentence1"])
tokenized_sentences_2 = tokenizer(raw_datasets["train"][14]["sentence2"])
input = (tokenized_sentences_1, tokenized_sentences_2)
print(input)

In [ ]:
input = tokenizer(
    raw_datasets["train"][14]["sentence1"],
    raw_datasets["train"][14]["sentence2"],
)
print(input)

In [ ]:
tokenized_dataset = tokenizer(
    raw_datasets["train"]["sentence1"],
    raw_datasets["train"]["sentence2"],
    padding=True,
    truncation=True,
)

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

In [ ]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

In [ ]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

In [ ]:
#Replicate the preprocessing on the GLUE SST-2 dataset.
from datasets import load_dataset

ds = load_dataset("gimmaru/glue-sst2")

ds

In [ ]:
ds["validation"][0]

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence"], padding="max_length", max_length=128, truncation=True)

In [ ]:
tokenized_datasets = ds.map(tokenize_function, batched=True)
tokenized_datasets

In [ ]:

tokenized_datasets = tokenized_datasets.remove_columns(["sentence", "idx"])




In [ ]:
samples = tokenized_datasets["validation"][:8]  # dict of lists

examples = []
for i in range(len(samples["input_ids"])):
    example = {k: v[i] for k, v in samples.items()}
    examples.append(example)

# Now collator will pad properly:
batch = data_collator(examples)


# FINE TUNING

In [ ]:
from huggingface_hub import login

login()

In [ ]:
!pip install -U datasets huggingface_hub fsspec

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("Yelp/yelp_review_full")
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")

def tokenize(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)

In [ ]:
small_train = dataset["train"].shuffle(seed=42).select(range(1000))
small_eval = dataset["test"].shuffle(seed=42).select(range(1000))

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-cased", num_labels=5)

In [ ]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

In [ ]:
!pip install evaluate

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # convert the logits to their predicted class
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/yelp_review_classifier",
    eval_strategy="epoch",
    push_to_hub=True,
    per_device_train_batch_size=4,
    max_steps=500,
    report_to ="none",
)


In [ ]:
from transformers import Trainer, TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
trainer.push_to_hub()